# 🎬 BENY-JOE IA — Génération Vidéo & Image HD
**Fondé par KHEDIM BENYAKHLEF dit BENY-JOE**

| Feature | Détail |
|---|---|
| 📐 Résolution | **1024×576 Full HD 16:9** |
| 🎬 Vidéo | **AnimateDiff + SD 1.5** |
| 🎤 Voix OFF | **gTTS FR/EN/AR** |
| 🎵 Musique | **MusicGen-medium IA** |
| ⚡ Device | **TPU v5 (JAX natif Kaggle)** |
| 🌐 Plateforme | **BENY-JOE IA sur Render** |

---
## ▶️ ORDRE D'EXÉCUTION :
1. **Cellule 0** — Vérification TPU (diagnostic)
2. **Cellule 1** — Secrets Kaggle (tokens)
3. **Cellule 2** — Installation des dépendances
4. **Cellule 3** — Init device + Chargement des modèles
5. **Cellule 4** — Serveur Flask + Tunnel ngrok
6. **Cellule 5** — Auto-push URL + surveillance vidéos

---
> ⚠️ **TPU Kaggle** : JAX est déjà installé nativement sur les notebooks Kaggle TPU.
> Ne jamais réinstaller `torch_xla` manuellement — cela casse l'environnement.
> Ce notebook utilise **JAX** pour la détection TPU et **PyTorch** pour SD 1.5 / AnimateDiff.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 — Vérification TPU (diagnostic rapide)                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import subprocess, sys

print("🔍 Vérification environnement Kaggle TPU...")
print()

# 1. JAX natif
try:
    import jax
    import jax.numpy as jnp
    devices = jax.devices()
    print(f"✅ JAX {jax.__version__} — {len(devices)} device(s) : {devices}")
    x = jnp.ones((512, 512))
    y = jnp.dot(x, x)
    print(f"✅ Calcul JAX OK — shape : {y.shape}")
except Exception as e:
    print(f"⚠️  JAX : {e}")
    print("   → Vérifiez que l'accélérateur Kaggle est bien réglé sur TPU")

print()

# 2. PyTorch
try:
    import torch
    print(f"✅ PyTorch {torch.__version__}")
except Exception as e:
    print(f"⚠️  PyTorch : {e}")

print()
print("👉 Si JAX et PyTorch sont OK → passez à la Cellule 1")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Chargement des secrets Kaggle                         ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Prérequis : ajoutez vos secrets dans Kaggle → Settings → Secrets
#   - GITHUB_TOKEN
#   - NGROK_TOKEN

import os

GITHUB_TOKEN = ""
NGROK_TOKEN  = ""

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    GITHUB_TOKEN = _secrets.get_secret("GITHUB_TOKEN")
    NGROK_TOKEN  = _secrets.get_secret("NGROK_TOKEN")
    print("✅ Tokens chargés depuis Kaggle Secrets !")
except Exception:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    NGROK_TOKEN  = os.environ.get("NGROK_TOKEN",  "")
    if GITHUB_TOKEN and NGROK_TOKEN:
        print("✅ Tokens chargés depuis les variables d'environnement")
    else:
        print("⚠️  Secrets non trouvés.")
        print("   → Ajoutez GITHUB_TOKEN et NGROK_TOKEN dans Kaggle → Settings → Secrets")
        # Décommentez et remplissez si besoin :
        # GITHUB_TOKEN = "ghp_votre_token_ici"
        # NGROK_TOKEN  = "2abc_votre_token_ici"

os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
os.environ["NGROK_TOKEN"]  = NGROK_TOKEN

print()
print(f"  GITHUB_TOKEN : {'✅ présent' if GITHUB_TOKEN else '❌ absent'}")
print(f"  NGROK_TOKEN  : {'✅ présent' if NGROK_TOKEN  else '❌ absent'}")
print()
print("👉 Passez à la Cellule 2")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Installation des dépendances (TPU Kaggle)             ║
# ║  ⚠️  NE PAS réinstaller torch ou torch_xla — déjà présents        ║
# ╚══════════════════════════════════════════════════════════════════════╝

import subprocess, sys

def pip_install(pkg, extra=""):
    cmd = f"{sys.executable} -m pip install -q {extra} {pkg}"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=900)
    status = "✅" if r.returncode == 0 else "⚠️"
    short  = pkg.split()[0][:50]
    print(f"  {status} {short}")
    if r.returncode != 0 and r.stderr:
        last_err = [l for l in r.stderr.strip().splitlines() if l.strip()][-1:]
        if last_err:
            print(f"     └─ {last_err[0][:120]}")
    return r.returncode == 0

print("╔══════════════════════════════════════════════╗")
print("║  BENY-JOE IA — Installation (Kaggle TPU)    ║")
print("╚══════════════════════════════════════════════╝")
print()

# ── 1. NumPy stable ───────────────────────────────────────────────────
print("🔧 Fix NumPy 1.26.4...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy==1.26.4", "--force-reinstall"],
    check=False, capture_output=True
)
print("  ✅ NumPy 1.26.4")
print()

# ── 2. Diffusion ──────────────────────────────────────────────────────
print("🔧 Diffusers / Transformers / Accelerate...")
for p in [
    "diffusers==0.28.2",
    "transformers==4.41.2",
    "accelerate==0.30.1",
    "safetensors==0.4.3",
    "omegaconf",
    "einops",
    "huggingface_hub==0.23.2",
    "compel",
]:
    pip_install(p)
print()

# ── 3. Vidéo & Image ──────────────────────────────────────────────────
print("🔧 Vidéo & Image...")
for p in [
    "imageio>=2.34.1",
    "imageio-ffmpeg",
    "opencv-python-headless",
    "Pillow>=10.3.0",
    "moviepy==1.0.3",
    "ffmpeg-python",
    "av",
]:
    pip_install(p)
print()

# ── 4. Audio ──────────────────────────────────────────────────────────
print("🔧 Audio...")
for p in ["pydub", "scipy", "soundfile", "gtts"]:
    pip_install(p)

print("  🎵 Audiocraft / MusicGen...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/facebookresearch/audiocraft.git"],
    capture_output=True, text=True, timeout=600
)
print(f"  {'✅' if r.returncode == 0 else '⚠️'} audiocraft")
print()

# ── 5. Serveur & Tunnel ───────────────────────────────────────────────
print("🔧 Flask + Ngrok...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "blinker==1.7.0", "--ignore-installed"],
    check=False, capture_output=True
)
for p in ["flask", "flask-cors", "pyngrok>=7.0.0", "requests", "python-dotenv"]:
    pip_install(p)
print()

# ── 6. Re-fix NumPy final ─────────────────────────────────────────────
print("🔧 Re-fix NumPy final...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy==1.26.4", "--force-reinstall"],
    check=False, capture_output=True
)
import numpy as np
print(f"  ✅ NumPy final : {np.__version__}")
print()
print("✅ Installation terminée — passez à la Cellule 3")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Init device + Chargement des modèles                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, gc, torch, numpy as np
from pathlib import Path

# ── Device ────────────────────────────────────────────────────────────
try:
    import jax
    _devs = jax.devices()
    print(f"✅ JAX TPU détecté : {_devs}")
    DEVICE = torch.device("cpu")   # AnimateDiff tourne sur CPU/TPU via JAX bridge
    DEVICE_NAME = f"TPU-v5 ({len(_devs)} cores)"
except Exception:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DEVICE_NAME = str(DEVICE)

os.environ["DEVICE_NAME"] = DEVICE_NAME
print(f"⚡ Device : {DEVICE_NAME}")
print()

# ── Chemins ────────────────────────────────────────────────────────────
OUTPUTS_DIR = "/kaggle/working/outputs"
MODELS_DIR  = "/kaggle/working/models"
Path(OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

# ── Chargement AnimateDiff ────────────────────────────────────────────
print("🔧 Chargement AnimateDiff + SD 1.5...")
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from diffusers.utils import export_to_video
from huggingface_hub import hf_hub_download

try:
    adapter = MotionAdapter.from_pretrained(
        "guoyww/animatediff-motion-adapter-v1-5-3",
        torch_dtype=torch.float16,
    )
    pipe_video = AnimateDiffPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        motion_adapter=adapter,
        torch_dtype=torch.float16,
    )
    pipe_video.scheduler = EulerDiscreteScheduler.from_config(
        pipe_video.scheduler.config,
        beta_schedule="linear",
    )
    pipe_video.enable_vae_slicing()
    pipe_video.enable_attention_slicing()
    pipe_video = pipe_video.to(DEVICE)
    print("✅ AnimateDiff chargé")
except Exception as e:
    print(f"⚠️  AnimateDiff : {e}")
    pipe_video = None

# ── Chargement SD 1.5 image ───────────────────────────────────────────
print("🔧 Chargement SD 1.5 image...")
from diffusers import StableDiffusionPipeline

try:
    pipe_image = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
    )
    pipe_image.enable_vae_slicing()
    pipe_image.enable_attention_slicing()
    pipe_image = pipe_image.to(DEVICE)
    print("✅ SD 1.5 image chargé")
except Exception as e:
    print(f"⚠️  SD 1.5 : {e}")
    pipe_image = None

# ── MusicGen ──────────────────────────────────────────────────────────
print("🔧 Chargement MusicGen...")
try:
    from audiocraft.models import MusicGen
    music_model = MusicGen.get_pretrained("facebook/musicgen-medium")
    music_model.set_generation_params(duration=10)
    print("✅ MusicGen chargé")
except Exception as e:
    print(f"⚠️  MusicGen : {e}")
    music_model = None

print()
print("=" * 52)
print(f"  BENY-JOE IA — Modèles chargés")
print(f"  Video   : {'✅' if pipe_video else '⚠️'}")
print(f"  Image   : {'✅' if pipe_image else '⚠️'}")
print(f"  MusicGen: {'✅' if music_model else '⚠️'}")
print("=" * 52)
print()
print("👉 Passez à la Cellule 4")

In [ ]:
# CELLULE 4 — Serveur Flask + Tunnel ngrok  [CORRIGE BENY-JOE]

import os, gc, time, uuid, logging, threading
import torch
from pathlib import Path
from datetime import datetime, timezone
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from gtts import gTTS

OUTPUTS_DIR  = '/kaggle/working/outputs'
FLASK_PORT   = 8765
RENDER_URL   = os.environ.get('RENDER_URL', 'https://benyjoe-ia.onrender.com')
SECRET_KEY   = os.environ.get('BENYJOE_SECRET', 'benyjoe-secret-2025')
DEVICE_NAME  = os.environ.get('DEVICE_NAME', 'Kaggle-TPU')
Path(OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO)
log = logging.getLogger('BENYJOE')
jobs = {}
lock = threading.Lock()

def set_job(jid, **kw):
    with lock:
        if jid not in jobs:
            jobs[jid] = {'id': jid, 'status': 'pending', 'progress': 0, 'step': '', 'result': None, 'error': None}
        jobs[jid].update(kw)

def get_ngrok_url():
    try:
        import requests as req
        r = req.get('http://localhost:4040/api/tunnels', timeout=5)
        for t in r.json().get('tunnels', []):
            if t.get('proto') == 'https':
                return t['public_url']
    except Exception:
        pass
    return os.environ.get('NGROK_URL', '')

def generate_voice(text, lang='fr', out_path=None):
    if out_path is None:
        out_path = os.path.join(OUTPUTS_DIR, f'voice_{uuid.uuid4().hex[:8]}.mp3')
    try:
        tts = gTTS(text=text[:500], lang=lang, slow=False)
        tts.save(out_path)
        return out_path
    except Exception as e:
        log.warning(f'gTTS : {e}')
        return None

def generate_music(prompt, duration=10, out_path=None):
    if out_path is None:
        out_path = os.path.join(OUTPUTS_DIR, f'music_{uuid.uuid4().hex[:8]}.wav')
    try:
        if music_model is None:
            return None
        music_model.set_generation_params(duration=min(duration, 30))
        wav = music_model.generate([prompt])
        import soundfile as sf
        audio_np = wav[0].cpu().numpy()
        if audio_np.ndim == 2:
            audio_np = audio_np[0]
        sf.write(out_path, audio_np, samplerate=32000)
        return out_path
    except Exception as e:
        log.warning(f'MusicGen : {e}')
        return None

def mix_audio_with_video(video_path, voice_path, music_path, out_path):
    try:
        from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip
        clip = VideoFileClip(video_path)
        audio_clips = []
        if voice_path and os.path.exists(voice_path):
            audio_clips.append(AudioFileClip(voice_path).volumex(1.0))
        if music_path and os.path.exists(music_path):
            audio_clips.append(AudioFileClip(music_path).volumex(0.35).set_duration(clip.duration))
        if audio_clips:
            clip = clip.set_audio(CompositeAudioClip(audio_clips))
        clip.write_videofile(out_path, codec='libx264', audio_codec='aac', logger=None, verbose=False)
        clip.close()
        return out_path
    except Exception as e:
        log.warning(f'moviepy mix : {e}')
        return video_path

def run_generation(jid, data):
    try:
        prompt   = data.get('prompt', '')
        gen_type = data.get('type', 'video')
        voice_on = data.get('voice', True)
        music_on = data.get('music', True)
        lang     = data.get('voice_lang', 'fr')
        mstyle   = data.get('music_style', 'cinematic')
        duration = int(data.get('duration', 10))
        resolution = data.get('resolution', '1024x576')
        fps      = int(data.get('fps', 24))
        frames   = int(data.get('frames', 32))
        w, h     = [int(x) for x in resolution.split('x')]

        set_job(jid, status='processing', progress=5, step='Initialisation...')

        if gen_type == 'image':
            set_job(jid, progress=20, step='Generation image SD 1.5...')
            result = pipe_image(prompt=prompt, negative_prompt='low quality, blurry',
                                height=h, width=w, num_inference_steps=40, guidance_scale=7.5)
            img = result.images[0]
            fname = f'BENYJOE_IMG_{jid}.png'
            img.save(os.path.join(OUTPUTS_DIR, fname), 'PNG')
            ngrok_base = get_ngrok_url()
            public_url = f'{ngrok_base}/outputs/{fname}' if ngrok_base else f'{RENDER_URL}/outputs/{fname}'
            set_job(jid, status='done', progress=100, step='Image prete !', result=public_url)
            try:
                import requests as req
                req.post(f'{RENDER_URL}/api/video-ready', json={'job_id': jid, 'video_url': public_url}, timeout=10)
            except: pass
            return

        set_job(jid, progress=15, step='Generation frames AnimateDiff...')
        num_frames = min(max(frames, 16), 128)
        output = pipe_video(prompt=prompt, negative_prompt='low quality, blurry',
                            height=min(h, 512), width=min(w, 512), num_frames=num_frames,
                            num_inference_steps=25, guidance_scale=7.5)
        frames_list = output.frames[0]

        set_job(jid, progress=55, step='Assemblage video...')
        raw_mp4 = os.path.join(OUTPUTS_DIR, f'raw_{jid}.mp4')
        from diffusers.utils import export_to_video
        export_to_video(frames_list, raw_mp4, fps=fps)

        set_job(jid, progress=65, step='Upscale HD...')
        hd_mp4 = os.path.join(OUTPUTS_DIR, f'hd_{jid}.mp4')
        import subprocess as sp
        sp.run(['ffmpeg', '-y', '-i', raw_mp4, '-vf', f'scale={w}:{h}:flags=lanczos',
                '-c:v', 'libx264', '-crf', '18', '-preset', 'fast', hd_mp4], capture_output=True)
        if not os.path.exists(hd_mp4):
            hd_mp4 = raw_mp4

        voice_path = None
        if voice_on:
            set_job(jid, progress=75, step='Generation voix OFF...')
            voice_path = generate_voice(prompt, lang=lang)

        music_path = None
        if music_on:
            set_job(jid, progress=82, step='Composition musicale...')
            music_desc = {'cinematic': 'epic cinematic orchestral score',
                          'electronic': 'electronic ambient synth music',
                          'ambient': 'calm ambient meditation music',
                          'epic': 'epic battle orchestral music',
                          'oriental': 'oriental arabic music oud'}.get(mstyle, 'cinematic music')
            music_path = generate_music(music_desc, duration=max(duration, 10))

        fname = f'BENYJOE_FINAL_{jid}.mp4'
        final_mp4 = os.path.join(OUTPUTS_DIR, fname)
        if voice_path or music_path:
            set_job(jid, progress=90, step='Mixage audio + video...')
            mix_audio_with_video(hd_mp4, voice_path, music_path, final_mp4)
        else:
            import shutil
            shutil.copy2(hd_mp4, final_mp4)

        for f in [raw_mp4, hd_mp4, voice_path, music_path]:
            if f and os.path.exists(f):
                try: os.remove(f)
                except: pass

        # ✅ FIX PRINCIPAL : URL publique complète via ngrok
        ngrok_base = get_ngrok_url()
        if ngrok_base:
            public_url = f'{ngrok_base}/outputs/{fname}'
        else:
            public_url = f'{RENDER_URL}/outputs/{fname}'

        set_job(jid, status='done', progress=100, step='Video prete !', result=public_url)
        print(f'Video URL : {public_url}')

        import requests as req
        try:
            req.post(f'{RENDER_URL}/api/video-ready', json={
                'job_id': jid, 'video_url': public_url, 'source': 'kaggle', 'device': DEVICE_NAME
            }, timeout=10)
            print(f'Render notifie : {public_url}')
        except Exception as e:
            print(f'Render notify erreur : {e}')

    except Exception as e:
        log.error(f'Generation {jid} : {e}')
        import traceback; traceback.print_exc()
        set_job(jid, status='error', error=str(e))

flask_app = Flask('BENYJOE_IA')
CORS(flask_app)

@flask_app.route('/health')
def health():
    return jsonify({'status': 'ok', 'platform': 'BENY-JOE IA',
                    'founder': 'KHEDIM BENYAKHLEF dit BENY-JOE', 'device': DEVICE_NAME,
                    'jobs': len(jobs), 'models': {'video': pipe_video is not None,
                    'image': pipe_image is not None, 'music': music_model is not None}})

@flask_app.route('/generate', methods=['POST'])
def generate():
    data = request.get_json(silent=True) or {}
    prompt = (data.get('prompt') or '').strip()
    if not prompt:
        return jsonify({'error': 'Prompt requis'}), 400
    jid = data.get('job_id') or str(uuid.uuid4())[:12]
    set_job(jid, status='queued', step='En file...', progress=0)
    t = threading.Thread(target=run_generation, args=(jid, data), daemon=True)
    t.start()
    return jsonify({'job_id': jid, 'status': 'queued'})

@flask_app.route('/api/jobs')
def api_jobs():
    with lock:
        return jsonify({'queue_size': len(jobs), 'jobs': dict(jobs)})

@flask_app.route('/api/jobs/<jid>')
def api_job(jid):
    with lock:
        job = jobs.get(jid)
    if not job:
        return jsonify({'error': 'Job introuvable'}), 404
    return jsonify(job)

@flask_app.route('/outputs/<path:filename>')
def serve_output(filename):
    return send_file(os.path.join(OUTPUTS_DIR, filename))

import threading as _th
_flask_thread = _th.Thread(
    target=lambda: flask_app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False),
    daemon=True, name='FlaskServer'
)
_flask_thread.start()
time.sleep(2)
print(f'Flask actif sur http://localhost:{FLASK_PORT}')

from pyngrok import ngrok, conf
NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    ngrok.kill()
    time.sleep(1)
    tunnel = ngrok.connect(FLASK_PORT, 'http')
    NGROK_URL = tunnel.public_url
    print(f'Tunnel ngrok : {NGROK_URL}')
    os.environ['NGROK_URL'] = NGROK_URL
    import requests as req
    try:
        r = req.post(f'{RENDER_URL}/api/kaggle-url', json={'url': NGROK_URL, 'secret': SECRET_KEY}, timeout=12)
        st = 'OK' if r.status_code in (200, 201) else f'Erreur ({r.status_code})'
        print(f'URL envoyee a Render : {st}')
    except Exception as e:
        print(f'Render non joignable : {e}')
else:
    print('NGROK_TOKEN absent - tunnel non cree')

print('='*52)
print('  BENY-JOE IA Cellule 4 operationnelle !')
print(f'  Flask  : http://localhost:{FLASK_PORT}')
print(f'  Render : {RENDER_URL}')
print('='*52)
print('Passez a la Cellule 5')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Auto-push URL + Watcher vidéos                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, time, json, base64, threading, requests
from datetime import datetime, timezone
import logging

log = logging.getLogger("BENYJOE-WATCHER")

# ════════ CONFIGURATION ═══════════════════════════════════════════════
# ⚠️  IMPORTANT : copiez l'URL de votre déploiement Render ici
RENDER_URL    = os.environ.get("RENDER_URL", "https://benyjoe-ia.onrender.com")
SECRET_KEY    = os.environ.get("BENYJOE_SECRET", "benyjoe-secret-2025")
GITHUB_TOKEN  = os.environ.get("GITHUB_TOKEN", "")
GITHUB_REPO   = os.environ.get("GITHUB_REPO", "")   # ex: username/repo
GITHUB_BRANCH = "main"
GITHUB_API    = "https://api.github.com"
OUTPUTS_DIR   = "/kaggle/working/outputs"
NGROK_URL     = os.environ.get("NGROK_URL", "")

# ════════════════════════════════════════════════════════════════════
def get_ngrok_url():
    """Récupère l'URL ngrok depuis le daemon local."""
    try:
        r = requests.get("http://localhost:4040/api/tunnels", timeout=5)
        for t in r.json().get("tunnels", []):
            if t.get("proto") == "https":
                return t["public_url"]
    except Exception:
        pass
    return NGROK_URL or None

def push_url_to_render(url):
    """Envoie l'URL ngrok à la plateforme Render."""
    try:
        r = requests.post(f"{RENDER_URL}/api/kaggle-url", json={
            "url": url, "secret": SECRET_KEY,
        }, timeout=12)
        return r.status_code in (200, 201)
    except Exception as e:
        print(f"  ⚠️ Render : {e}")
        return False

def upload_video_to_github(video_path, job_id=None):
    """Upload une vidéo dans les releases GitHub (optionnel)."""
    if not GITHUB_TOKEN or not GITHUB_REPO or not os.path.exists(video_path):
        return None
    headers  = {"Authorization": f"token {GITHUB_TOKEN}",
                "Accept":        "application/vnd.github.v3+json"}
    api_base = f"{GITHUB_API}/repos/{GITHUB_REPO}"
    tag      = "benyjoe-ia-videos"
    try:
        r = requests.get(f"{api_base}/releases/tags/{tag}", headers=headers, timeout=10)
        if r.status_code == 200:
            upload_url = r.json()["upload_url"].split("{")[0]
        else:
            r2 = requests.post(f"{api_base}/releases", headers=headers, timeout=15,
                               json={"tag_name": tag, "name": "BENY-JOE IA Videos",
                                     "body": "Vidéos BENY-JOE IA", "draft": False})
            if r2.status_code != 201:
                return None
            upload_url = r2.json()["upload_url"].split("{")[0]
        ts         = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        asset_name = f"{ts}_{os.path.basename(video_path)}"
        with open(video_path, "rb") as fh:
            video_bytes = fh.read()
        r3 = requests.post(
            f"{upload_url}?name={asset_name}",
            headers={"Authorization": f"token {GITHUB_TOKEN}", "Content-Type": "video/mp4"},
            data=video_bytes, timeout=300
        )
        if r3.status_code == 201:
            return r3.json()["browser_download_url"]
    except Exception as e:
        print(f"  ⚠️ GitHub upload : {e}")
    return None

def notify_render(video_url, job_id=None):
    """Notifie Render qu'une vidéo est prête."""
    try:
        requests.post(f"{RENDER_URL}/api/video-ready", json={
            "job_id":    job_id or "unknown",
            "video_url": video_url,
            "source":    "kaggle-watcher",
            "device":    os.environ.get("DEVICE_NAME", "Kaggle-TPU"),
        }, timeout=12)
    except: pass

# ── Watcher daemon ────────────────────────────────────────────────────
_watched     = set()
_watcher_run = True

def auto_push_watcher(interval=15):
    while _watcher_run:
        try:
            if os.path.isdir(OUTPUTS_DIR):
                for fname in list(os.listdir(OUTPUTS_DIR)):
                    if not fname.endswith(".mp4") or fname in _watched:
                        continue
                    fpath = os.path.join(OUTPUTS_DIR, fname)
                    try:    sz1 = os.path.getsize(fpath)
                    except: continue
                    time.sleep(3)
                    try:    sz2 = os.path.getsize(fpath)
                    except: continue
                    if sz1 != sz2 or sz2 == 0:
                        continue
                    _watched.add(fname)
                    jid = fname.replace("BENYJOE_FINAL_", "").replace(".mp4", "")
                    print(f"\n🎬 Nouvelle vidéo détectée : {fname}")
                    dl_url = upload_video_to_github(fpath, jid)
                    if dl_url:
                        notify_render(dl_url, job_id=jid)
                        print(f"  ✅ Notifié Render : {dl_url}")
        except Exception as e:
            log.error(f"Watcher : {e}")
        time.sleep(interval)

# ── Exécution ─────────────────────────────────────────────────────────
print("=" * 52)
print("  BENY-JOE IA — Cellule 5 : Auto-Push & Watcher")
print("=" * 52)
print()

# Re-push URL ngrok
print("🔍 Recherche URL ngrok active...")
ngrok_url = get_ngrok_url()
render_ok  = False
if ngrok_url:
    print(f"  ✅ URL : {ngrok_url}")
    render_ok = push_url_to_render(ngrok_url)
    print(f"  Render : {'✅ OK' if render_ok else '⚠️  Échec'}")
else:
    print("  ⚠️  Aucun tunnel ngrok actif — relancez la Cellule 4")

# Démarrer le watcher
watcher_thread = threading.Thread(
    target=auto_push_watcher, args=(15,), daemon=True, name="BenyJoeWatcher"
)
watcher_thread.start()

print()
print("=" * 52)
print("Récapitulatif :")
print(f"  🌐 Render URL   : {RENDER_URL}")
print(f"  🔗 ngrok URL    : {ngrok_url or 'non disponible'}")
print(f"  ✉️  Push Render  : {'✅ OK' if render_ok else '⚠️'}")
print(f"  👁️  Watcher     : ✅ actif (toutes les 15s)")
print(f"  📂 Surveille    : {OUTPUTS_DIR}")
print()
print("✅ BENY-JOE IA 100% opérationnel — fondé par KHEDIM BENYAKHLEF dit BENY-JOE !")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELLULE DIAGNOSTIC — Vérification complète (facultative)          ║
# ╚══════════════════════════════════════════════════════════════════════╝

import requests, json

print("🔍 Diagnostic BENY-JOE IA...")
print()

# 1. Flask local
try:
    r    = requests.get("http://localhost:8765/health", timeout=5)
    data = r.json()
    print("✅ Flask actif :")
    for k, v in data.items():
        print(f"   {str(k):30s}: {v}")
except Exception as e:
    print(f"❌ Flask non actif : {e}")
    print("   👉 Relancez la Cellule 4")

print()

# 2. Jobs
try:
    r    = requests.get("http://localhost:8765/api/jobs", timeout=5)
    data = r.json()
    print(f"📋 Jobs : {data.get('queue_size', 0)} en traitement")
    for jid, state in list((data.get('jobs') or {}).items())[:5]:
        print(f"   {jid} → {state.get('status')} ({state.get('progress')}%) {state.get('step','')}")
except Exception as e:
    print(f"⚠️  Jobs : {e}")

print()

# 3. Ngrok
try:
    r       = requests.get("http://localhost:4040/api/tunnels", timeout=5)
    tunnels = r.json().get("tunnels", [])
    if tunnels:
        print("✅ Ngrok actif :")
        for t in tunnels:
            print(f"   {t.get('proto'):5s} → {t.get('public_url')}")
    else:
        print("⚠️  Ngrok : aucun tunnel")
except Exception as e:
    print(f"⚠️  Ngrok : {e}")

print()
print("✅ Diagnostic terminé — BENY-JOE IA by KHEDIM BENYAKHLEF")
